# 03 - Feature Engineering

## Import

In [3]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
import pyarrow
from sqlalchemy import create_engine
from dotenv import load_dotenv

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

import warnings
warnings.filterwarnings('ignore')

## Load dataset

In [4]:
SAVE_PATH = r"C:\Users\Admin\Downloads\processed\processed"

df_category = pl.read_parquet(f"{SAVE_PATH}/category.parquet")
df_seller = pl.read_parquet(f"{SAVE_PATH}/seller.parquet")
df_product = pl.read_parquet(f"{SAVE_PATH}/product.parquet")
df_price = pl.read_parquet(f"{SAVE_PATH}/price_offer.parquet")
df_review = pl.read_parquet(f"{SAVE_PATH}/review.parquet")
df_reviewer = pl.read_parquet(f"{SAVE_PATH}/reviewer.parquet")
df_service = pl.read_parquet(f"{SAVE_PATH}/service.parquet")
df_coupon = pl.read_parquet(f"{SAVE_PATH}/coupon.parquet")
df_offer_coupon = pl.read_parquet(f"{SAVE_PATH}/offer_coupon.parquet")
df_offer_service = pl.read_parquet(f"{SAVE_PATH}/offer_service.parquet")

In [22]:
PB_OUT = os.path.join(SAVE_PATH, "powerbi")
os.makedirs(PB_OUT, exist_ok=True)

## Mục tiêu phân tích

### Phân tích và Dự đoán Giá & Khuyến mãi (Pricing & Discount)

**Mục tiêu 1:** Phân tích sự phân tán của tỷ lệ giảm giá (discount_percent) trên top 5 danh mục sản phẩm phổ biến nhất để xác định "khoảng giá trị trung vị" (sweet spot) giúp duy trì lượt đánh giá cao nhất, nhằm đề xuất khung định giá tối ưu trước ngày 01/04/2026.

In [11]:
latest_price = df_price.sort("crawl_time").group_by("product_id").agg(
    pl.col("current_price").last(),
    pl.col("original_price").last(),
    pl.col("discount_percent").last(),
    pl.col("offer_id").last(),
    pl.col("crawl_time").last(),
)

cat_l1_mapping = df_category.with_columns(
    pl.col("category_path")
    .str.split(">")
    .list.get(0)
    .str.strip_chars()
    .alias("l1_category_name")
).select(["category_id", "l1_category_name"])

top5_l1_names = (
    df_product.join(cat_l1_mapping, on="category_id", how="left")
    .group_by("l1_category_name")
    .len()
    .sort("len", descending=True)
    .head(5)["l1_category_name"].to_list()
)

pbi_pricing_detail = (
    latest_price.join(
        df_product.select(["product_id", "category_id", "review_score", "review_count", "sold_quantity"]),
        on="product_id",
        how="inner",
    )
    .join(cat_l1_mapping, on="category_id", how="left")
    .filter(pl.col("l1_category_name").is_in(top5_l1_names))
    .with_columns(
        ((pl.col("discount_percent") / 5.0).floor() * 5.0).clip(0, 100).alias("discount_bin_pct"),
        (pl.col("review_score") * (pl.col("review_count").cast(pl.Float64) + 1.0).log1p()).alias("sweet_score_weighted"),
    )
)

pbi_pricing_summary = (
    pbi_pricing_detail.group_by(["l1_category_name", "discount_bin_pct"])
    .agg(
        pl.col("review_score").median().alias("median_review_score"),
        pl.col("review_count").sum().alias("sum_review_count"),
        pl.col("sweet_score_weighted").median().alias("median_sweet_score"),
        pl.len().alias("n_products"),
        pl.col("discount_percent").median().alias("median_discount_pct"),
        pl.col("current_price").median().alias("median_current_price"),
    )
    .with_columns(
        pl.col("median_sweet_score")
        .rank(method="ordinal", descending=True)
        .over("l1_category_name")
        .alias("sweet_rank_in_l1")
    )
    .with_columns(
        (pl.col("sweet_rank_in_l1") == 1).alias("is_sweet_spot_bin")
    )
    .sort(["l1_category_name", "discount_bin_pct"])
)

pbi_pricing_detail.write_parquet(os.path.join(PB_OUT, "pbi_pricing_discount_detail.parquet"))
pbi_pricing_summary.write_parquet(os.path.join(PB_OUT, "pbi_pricing_discount_summary.parquet"))

**Mục tiêu 2:** Thống kê tỷ lệ áp dụng chồng chéo các mã giảm giá (coupon) trên toàn bộ sản phẩm để nhận diện 20% các loại mã kém hiệu quả nhất, từ đó đề xuất cấu trúc lại ngân sách khuyến mãi cho nền tảng trước ngày 30/04/2026.

In [12]:
_offer_prod = df_price.select(["offer_id", "product_id"]).unique()
cup_on_prod = (
    df_offer_coupon.join(_offer_prod, on="offer_id", how="inner")
    .select(["coupon_id", "product_id"])
    .unique()
)

n_scope = df_product.select(pl.col("product_id").n_unique()).item()
_overlap_rows = cup_on_prod.group_by("product_id").agg(pl.col("coupon_id").n_unique().alias("n_coupons"))
pbi_coupon_overlap_products = _overlap_rows.with_columns(
    (pl.col("n_coupons") > 1).alias("has_stacked_coupons")
)

_stack_rate = float(
    pbi_coupon_overlap_products.filter(pl.col("has_stacked_coupons")).height
    / max(pbi_coupon_overlap_products.height, 1)
)

_n_stacked = pbi_coupon_overlap_products.filter(pl.col("has_stacked_coupons")).height
pbi_coupon_kpis = pl.DataFrame(
    {
        "n_products_in_catalog": [n_scope],
        "n_products_with_any_coupon": [pbi_coupon_overlap_products.height],
        "share_products_stacked_among_coupon_skus": [_stack_rate],
        "share_products_stacked_vs_all_catalog": [
            float(_n_stacked / max(n_scope, 1))
        ],
    }
)

coupon_reach = cup_on_prod.group_by("coupon_id").agg(
    pl.len().alias("n_distinct_products"),
    (pl.len().cast(pl.Float64) / pl.lit(max(n_scope, 1))).alias("reach_share_of_catalog"),
)

worst_n = max(int(coupon_reach.height * 0.2), 1)
coupon_reach_sorted = coupon_reach.sort("n_distinct_products")
worst_ids = coupon_reach_sorted.head(worst_n)["coupon_id"].to_list()
pbi_coupon_efficiency = (
    coupon_reach.with_columns(
        pl.col("coupon_id").is_in(worst_ids).alias("is_bottom_20pct_low_reach")
    )
    .join(
        df_coupon.select(["coupon_id", "coupon_code", "title"]),
        on="coupon_id",
        how="left",
    )
)
pbi_coupon_overlap_products.write_parquet(os.path.join(PB_OUT, "pbi_coupon_overlap_by_product.parquet"))
pbi_coupon_kpis.write_parquet(os.path.join(PB_OUT, "pbi_coupon_stacking_kpis.parquet"))
pbi_coupon_efficiency.write_parquet(os.path.join(PB_OUT, "pbi_coupon_efficiency_tiers.parquet"))

### Phân tích Phản hồi Khách hàng (NLP & Sentiment Analysis)

**Mục tiêu 1:** Phân tích tần suất từ khóa (Text Mining) trên tập hợp đánh giá 1-2 sao để định lượng 3 nguyên nhân cốt lõi gây thất vọng lớn nhất, nhằm đề xuất giải pháp vận hành giúp giảm 15% tỷ lệ phàn nàn của khách hàng trước ngày 01/04/2026.

In [13]:
_low = df_review.filter(pl.col("rating_score").is_in([1, 2])).select(
    ["review_id", "product_id", "rating_score", "review_content", "thank_count"]
)

vietnamese_chars = r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ\s]"
_low_txt = _low.with_columns(
    pl.col("review_content")
    .fill_null("")
    .str.to_lowercase()
    .str.replace_all(vietnamese_chars, " ")
    .alias("txt_norm")
)

pbi_review_low_theme = _low_txt.with_columns(
    pl.col("txt_norm").str.contains(r"giao|vận chuyển|ship|shipper|bưu|nhận hàng|chậm|đóng gói").alias("hit_delivery"),
    pl.col("txt_norm").str.contains(r"lỗi|hỏng|kém|chất lượng|fake|nhái|đểu|đổi trả|bảo hành|rách|xước").alias("hit_quality"),
    pl.col("txt_norm").str.contains(r"giá|đắt|mắc|sale|khuyến mãi|giảm|hoàn tiền|lừa").alias("hit_price"),
)

pbi_nlp_theme_summary = pl.DataFrame(
    {
        "theme_key": ["delivery_logistics", "quality_trust", "price_promo"],
        "theme_label_vn": ["Giao hàng / vận chuyển", "Chất lượng / niềm tin", "Giá / khuyến mãi"],
        "n_reviews_flagged": [
            int(pbi_review_low_theme["hit_delivery"].sum()),
            int(pbi_review_low_theme["hit_quality"].sum()),
           int(pbi_review_low_theme["hit_price"].sum()),
        ],
    }
).with_columns(
    (pl.col("n_reviews_flagged") / pl.lit(max(pbi_review_low_theme.height, 1))).alias("share_of_low_rating_reviews")
)

_expl = (
    pbi_review_low_theme.with_columns(pl.col("txt_norm").str.split(" ").alias("_tok"))
    .explode("_tok")
    .filter(pl.col("_tok").str.len_bytes() > 2)
    .rename({"_tok": "token"})
)

_stop = {
    "và", "của", "là", "có", "không", "một", "cho", "với", "đã", "này", "được",
    "thì", "ở", "nhưng", "tôi", "quá", "rất", "cũng", "trên", "khi", "sẽ", "như", "lại", "mà", "vẫn",
    "hàng", "shop", "sản", "phẩm", "mua", "nhận", "mình", "thấy", "về", "đặt", "giao", "sp"
}

pbi_nlp_low_rating_keywords = (
    _expl.filter(~pl.col("token").is_in(list(_stop)))
    .group_by("token")
    .len()
    .sort("len", descending=True)
    .head(50)
    .rename({"len": "token_count"})
)

pbi_review_low_theme.write_parquet(os.path.join(PB_OUT, "pbi_review_low_rating_themes.parquet"))
pbi_nlp_theme_summary.write_parquet(os.path.join(PB_OUT, "pbi_review_low_rating_theme_summary.parquet"))
pbi_nlp_low_rating_keywords.write_parquet(os.path.join(PB_OUT, "pbi_review_low_rating_keywords.parquet"))

**Mục tiêu 2:** Đo lường hệ số tương quan giữa độ dài văn bản đánh giá (số lượng từ) và điểm rating thực tế trên mẫu 100.000 bình luận để xây dựng quy tắc lọc tự động, nhằm đẩy 5 đánh giá hữu ích nhất lên top trang sản phẩm trước ngày 05/05/2026.

In [14]:
_review_sample_corr = df_review.filter(pl.col("review_content").is_not_null()).sample(
    n=min(100_000, df_review.height), seed=42
)
_wc = _review_sample_corr.with_columns(
    pl.col("review_content")
    .fill_null("")
    .str.split(" ")
    .list.len()
    .alias("word_count")
)

# Tính tương quan với Rating
_cm_rating = np.corrcoef(_wc["word_count"].to_numpy(), _wc["rating_score"].to_numpy().astype(float))
_r_rating = float(_cm_rating[0, 1]) if not np.isnan(_cm_rating[0, 1]) else float("nan")

# Tính tương quan với Thank Count (Độ hữu ích)
_cm_thank = np.corrcoef(_wc["word_count"].to_numpy(), _wc["thank_count"].to_numpy().astype(float))
_r_thank = float(_cm_thank[0, 1]) if not np.isnan(_cm_thank[0, 1]) else float("nan")

pbi_review_length_corr = pl.DataFrame({
    "pearson_word_count_vs_rating": [_r_rating],
    "pearson_word_count_vs_thanks": [_r_thank],
    "sample_size": [_wc.height],
})

_wc.write_parquet(os.path.join(PB_OUT, "pbi_review_length_sample_50k.parquet"))
pbi_review_length_corr.write_parquet(os.path.join(PB_OUT, "pbi_review_length_correlation.parquet"))

### Phân cụm và Đánh giá Nhà bán hàng (Seller Performance)

**Mục tiêu 1:** Phân tích dữ liệu về lượng reviews và rating hiện tại để phân cụm (Clustering) 100% gian hàng thành 3 nhóm năng lực (Tốt, Khá, Cần cải thiện), từ đó đề xuất chiến lược hỗ trợ tương ứng giúp tăng 15% độ uy tín cho nhóm "Cần cải thiện" trước ngày 25/04/2026.

In [15]:
pbi_seller_tiers = (
    df_seller.with_columns(
        (
            pl.col("seller_rating").clip(0.01, 5.0)
            * (pl.col("total_reviews").cast(pl.Float64) + 1.0).log1p()
    ).alias("perf_score"),
        pl.col("total_reviews").alias("follower_engagement_proxy"),
    )
    .with_columns(
        ((pl.col("perf_score").rank(method="average") - 0.5) / pl.lit(float(max(df_seller.height, 1)))).alias("_pct")
    )
    .with_columns(
        pl.when(pl.col("_pct") <= 1.0 / 3.0)
        .then(pl.lit("Cần cải thiện"))
        .when(pl.col("_pct") <= 2.0 / 3.0)
        .then(pl.lit("Khá"))
        .otherwise(pl.lit("Tốt"))
        .alias("seller_tier_vn")
    )
    .drop(["_pct"])
)

pbi_seller_tiers.write_parquet(os.path.join(PB_OUT, "pbi_seller_performance_tiers.parquet"))

TÍNH TỶ LỆ BIẾN ĐỘNG GIÁ PHỤC VỤ CHO MỤC TIÊU PHÂN TÍCH 1 (CỦA THƯƠNG)

In [9]:
# lấy giá mới nhất của mỗi product
df_price_latest = (
    df_price
    .filter(
        pl.col("product_id").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .sort(["product_id", "crawl_time"])
    .group_by("product_id")
    .tail(1)
    .select(["product_id", "current_price"])
)

df = (
    df_product
    .select(["product_id", "category_id"])
    .join(
        df_category.select(["category_id", "category_name"]),
        on="category_id",
        how="left"
    )
    .join(
        df_price_latest,
        on="product_id",
        how="left"
    )
    .filter(
        pl.col("category_name").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .with_columns(
        pl.col("current_price").log1p().alias("log_price")
    )
)

# Tính grand mean
grand_mean = df.select(pl.col("log_price").mean()).item()

# Tính SST = tổng biến thiên toàn bộ
sst = df.select(
    ((pl.col("log_price") - grand_mean) ** 2).sum().alias("sst")
).item()

# Tính mean theo từng ngành
group_means = (
    df.group_by("category_name")
    .agg(pl.col("log_price").mean().alias("group_mean"))
)

# Tính SSE = tổng biến thiên còn lại sau khi giải thích bởi ngành hàng
df_with_mean = df.join(group_means, on="category_name", how="left")

sse = df_with_mean.select(
    ((pl.col("log_price") - pl.col("group_mean")) ** 2).sum().alias("sse")
).item()

# R-squared
r2 = 1 - sse / sst

print(f"R-squared = {r2:.4f}")


R-squared = 0.6458
